# DeBERTa-v3-large — benchmark CÔNG BẰNG (DDP + fp16, multi-seed)

Train + đánh giá **microsoft/deberta-v3-large** cho bài toán phát hiện văn bản AI, chạy trên **Kaggle T4×2** (DDP, 2 process).

**Đã hợp nhất cấu hình cho cả 3 model (RoBERTa / DeBERTa-v3 / ModernBERT)** để so sánh công bằng: cả 3 dùng CHUNG một `train_ddp.py` **giống hệt nhau**, chỉ khác đúng khối CONFIG (tên model + `PER_DEVICE_BATCH` để vừa VRAM). Nhờ đó chênh lệch kết quả quy được về **kiến trúc** — đạt chuẩn thí nghiệm có kiểm soát.

**Giống nhau cả 3 model:** optimizer `adamw_bnb_8bit` (AdamW 8-bit), `fp16` + DDP, LR `1e-5`, warmup `0.06`, adam_epsilon `1e-8`, weight_decay `0.01`, **effective batch = 32**, `max_len 512`, 10 epoch + early-stop patience 2, và `model_accepts_loss_kwargs=False` (chuẩn hoá gradient-accumulation đúng — sửa lỗi LR-×2).

**Chỉ khác (không ảnh hưởng động lực học):** kiến trúc model, `MODEL_KWARGS` (chỉ là tham số load), và `PER_DEVICE_BATCH` (nút chỉnh VRAM; đã bù grad-accum → effective batch vẫn 32, không đổi kết quả).

> ⚠️ **DeBERTa-v3-large**: nếu gặp NaN/OOM dưới fp16, đặt `FORCE_FP32=True` (chỉ notebook này) — lưu ý điều đó phá tính đồng bộ precision với 2 model kia. Nên chạy **SMOKE trước** để chắc chắn ổn.

**Quy trình:** Run cell 0 (cài đặt) → cell ghi `train_ddp.py` → *(tuỳ chọn)* SMOKE → **FULL RUN nhiều seed** → **Tổng hợp mean ± std**.

In [ ]:
!pip install -q -U "transformers==4.57.6" datasets accelerate scikit-learn sentencepiece bitsandbytes

In [ ]:
%%writefile train_ddp.py
# train_ddp.py — DeBERTa-v3-large trên DUAL T4 bằng DDP.
# Chạy: OMP_NUM_THREADS=2 torchrun --standalone --nproc_per_node=2 train_ddp.py
# Smoke DDP nhanh (verify 2 GPU bắt tay, ~vài phút): SMOKE=1 OMP_NUM_THREADS=2 torchrun --standalone --nproc_per_node=2 train_ddp.py
#
# Vì sao DDP (không phải DataParallel / device_map):
#   - DDP: mỗi GPU 1 process, chỉ ALL-REDUCE gradient -> ~1.7-1.8x thật, không replicate model mỗi step.
#   - grad_accum chia theo WORLD_SIZE => effective batch VẪN = 32 => LR/động lực học y hệt bản 1-GPU đã verify.
#   - Giữ nguyên fix model_accepts_loss_kwargs=False; DDP tự average gradient giữa 2 process.
import os, json, random, warnings
from pathlib import Path

# Đặt allocator config trước khi import torch/CUDA.
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score, average_precision_score

from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          Trainer, TrainingArguments, EarlyStoppingCallback)
from datasets import Dataset

warnings.filterwarnings("ignore")

# ── CONFIG (khớp train_02 bản 1-GPU đã verify) ───────────────────
NB_VERSION   = "unified-ddp-fp16-adamw8bit-multiseed-2026-07-23-deberta-v3-large"
RUN_NAME     = "deberta_v3_large"
MODEL_NAME   = "microsoft/deberta-v3-large"
MODEL_KWARGS = {}   # kwargs load riêng theo model (không ảnh hưởng tối ưu hoá)
LEARNING_RATE = 1e-5
MAX_LEN          = 512
EFFECTIVE_BATCH  = 32
PER_DEVICE_BATCH = 2                    # chỉ để vừa VRAM; grad-accum bù -> effective batch = 32 (động lực học không đổi)
                        # 2 x 2 GPU x accum 8 = effective batch 32, không đổi động lực học.
EPOCHS           = 10   # giữ nguyên yêu cầu 10 epoch; early stopping vẫn có thể dừng sớm.
EARLY_STOP_PATIENCE = 2
SEED             = int(os.environ.get("RUN_SEED", "42"))  # đổi qua env RUN_SEED để chạy nhiều seed
FORCE_FP32       = False   # AMP FP16 speed test; chỉ chạy full nếu smoke không NaN/OOM.
ADAM_EPSILON     = 1e-8               # ĐỒNG BỘ (mặc định AdamW)
WARMUP_RATIO     = 0.06               # ĐỒNG BỘ
MAX_GRAD_NORM    = 1.0
GRAD_CKPT        = False              # ĐỒNG BỘ tắt cả 3 (8-bit đủ nhẹ VRAM -> tắt để nhanh, result-neutral)
OPTIM        = "adamw_bnb_8bit"       # ĐỒNG BỘ 3 model (AdamW 8-bit: nhanh + nhẹ VRAM, chất lượng ~ đủ chính xác)
TARGET_FPR       = 0.01
# ── Bản dataset dùng để train/eval ────────────────────────────────────
#   "deconf2"    = đã khử confound bề mặt v2 (02c_deconfound_v2.ipynb)  <- ĐANG DÙNG
#                  canonicalize + truncate_pair_sentences + lọc notation_gap 85%.
#                  Đo bằng shortcut_audit.py (22 đặc trưng, 4 tầng):
#                    tầng process (dấu vết quy trình) 0.5055  <- cổng cứng, PASS
#                    tổng 22 đặc trưng               0.8909
#                  7.025 cặp train (85% của 8.265) — lọc bỏ những cặp mà bản AI lược
#                  ký hiệu/số/tên riêng của bài gốc nhiều nhất.
#   "deconf"     = bản 02b_deconfound.ipynb. ĐỪNG dùng cho bản chính thức:
#                  02b tự báo "surface-GBM 0.6867, G6 PASS" nhưng con số đó chỉ đo 6
#                  đặc trưng, không có avg_word_len (0.7795) lẫn type_token_ratio
#                  (0.7727). Đo đủ 22: 0.9114. Và truncate_pair cắt thô theo từ làm
#                  ~nửa văn bản kết thúc giữa câu -> word_count 0.5000 + ends_terminal
#                  0.5236 khi đứng riêng nhưng GHÉP LẠI = 0.7264.
#   "normalized" = bản GỐC, CÒN shortcut: chỉ ĐẾM SỐ TỪ đã đoán đúng human/AI
#                  ở AUROC 0.7193 (surface-GBM 0.8080). G6 FAIL.
#
# PHẢI đặt TRONG train_ddp.py, KHÔNG phải trong notebook: file này chạy qua
# torchrun ở tiến trình RIÊNG nên biến của notebook không tới được. Đặt ngoài thì
# script chết NameError — và trước đó cell còn không chạy nổi, vì dòng
# `DATASET_TAG = ...` ở trên đẩy %%writefile khỏi dòng đầu, làm Jupyter hiểu nó
# thành line magic: "Line magic function `%%writefile` not found".
#
# LƯU Ý: Kaggle dataset phải chứa dataset_{train,val,test_ood_model,test_fp}_deconf2.parquet
#        VÀ matched_eval_{val,test_ood_model}_deconf2.parquet (đánh giá đối-shortcut).
#        Danh sách file cần upload: KAGGLE_UPLOAD.md ở gốc repo.
DATASET_TAG = "deconf2"

SMOKE_TEST          = os.environ.get("SMOKE", "0") == "1"
SMOKE_MAX_STEPS     = 60
SMOKE_N             = 512
SMOKE_VAL_PER_CLASS = 128

KAGGLE_INPUT = "/kaggle/input/datasets/<kaggle-user>/halfdatadetector"

# ── Distributed env (torchrun đặt sẵn) ───────────────────────────
WORLD_SIZE = int(os.environ.get("WORLD_SIZE", "1"))
LOCAL_RANK = int(os.environ.get("LOCAL_RANK", "0"))
RANK       = int(os.environ.get("RANK", "0"))
IS_MAIN    = RANK == 0

if WORLD_SIZE != 2:
    raise RuntimeError(
        f"Notebook này yêu cầu DDP 2 GPU nhưng WORLD_SIZE={WORLD_SIZE}. "
        "Hãy chạy bằng torchrun --standalone --nproc_per_node=2 train_ddp.py"
    )

def log(*a):
    if IS_MAIN:
        print(*a, flush=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
USE_FP16 = torch.cuda.is_available() and not FORCE_FP32
grad_accum = max(1, EFFECTIVE_BATCH // (PER_DEVICE_BATCH * WORLD_SIZE))
actual_effective_batch = PER_DEVICE_BATCH * WORLD_SIZE * grad_accum
assert actual_effective_batch == EFFECTIVE_BATCH, (
    f"Effective batch lệch: {actual_effective_batch} != {EFFECTIVE_BATCH}"
)

log(f"### NB_VERSION={NB_VERSION} | WORLD_SIZE={WORLD_SIZE} | SMOKE={SMOKE_TEST} ###")
log(f"per_device={PER_DEVICE_BATCH} x world={WORLD_SIZE} x accum={grad_accum} "
    f"= effective {PER_DEVICE_BATCH * WORLD_SIZE * grad_accum}  (kỳ vọng 32)")
log(f"USE_FP16={USE_FP16}  GRAD_CKPT={GRAD_CKPT}  optim={OPTIM}  max_len={MAX_LEN}")

# ── Tìm thư mục parquet ──────────────────────────────────────────
def find_data_dir():
    target = f"dataset_train_{DATASET_TAG}.parquet"
    if (Path(KAGGLE_INPUT) / target).exists():
        return Path(KAGGLE_INPUT)
    for base in [Path(KAGGLE_INPUT), Path("output"), Path("."), Path("/kaggle/input")]:
        if base.exists():
            hits = list(base.rglob(target))
            if hits:
                return hits[0].parent
    raise FileNotFoundError(f"Không tìm thấy {target}. Kiểm tra KAGGLE_INPUT = {KAGGLE_INPUT}")

DATA_DIR = find_data_dir()
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
OUT_DIR = OUT_DIR / f"artifacts_{RUN_NAME}_seed{SEED}"
if IS_MAIN:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
log("DATA_DIR :", DATA_DIR)
log("OUT_DIR  :", OUT_DIR)

# ── Melt cặp wide -> long ────────────────────────────────────────
def melt_pairs(df):
    h = pd.DataFrame({"text": df["human_text"], "label": 0, "paper_id": df["paper_id"],
                      "ai_degree": 0, "generator_family": "human",
                      "sample_weight": 1.0})
    a = pd.DataFrame({"text": df["ai_text"], "label": 1, "paper_id": df["paper_id"],
                      "ai_degree": df.get("ai_degree", 3),
                      "generator_family": df.get("generator_family", "unknown"),
                      "sample_weight": 1.0})
    out = pd.concat([h, a], ignore_index=True)
    return out[out["text"].notna() & (out["text"].astype(str).str.len() > 0)].reset_index(drop=True)

def melt_fp(df):
    has_ai = df["ai_text"].notna() if "ai_text" in df.columns else pd.Series(False, index=df.index)
    text = df["ai_text"].where(has_ai, df["human_text"]) if "ai_text" in df.columns else df["human_text"]
    out = pd.DataFrame({"text": text, "label": 0, "paper_id": df["paper_id"],
                        "ai_degree": df.get("ai_degree", 0).fillna(0).astype(int) if "ai_degree" in df.columns else 0,
                        "generator_family": "human", "sample_weight": 1.0})
    return out[out["text"].notna() & (out["text"].astype(str).str.len() > 0)].reset_index(drop=True)

def load(name, fp_mode=False):
    p = DATA_DIR / f"dataset_{name}_{DATASET_TAG}.parquet"
    if not p.exists():
        log(f"  ⚠️  thiếu {p.name} — bỏ qua"); return None
    df = pd.read_parquet(p)
    return melt_fp(df) if fp_mode else melt_pairs(df)

train_df = load("train"); val_df = load("val")
ood_df = load("test_ood_model"); fp_df = load("test_fp", fp_mode=True)
for nm, d in [("train", train_df), ("val", val_df), ("test_ood_model", ood_df), ("test_fp", fp_df)]:
    if d is not None:
        log(f"{nm:<16} {len(d):>7,} mẫu  label={d['label'].value_counts().to_dict()}")

if SMOKE_TEST and val_df is not None:
    val_df = (val_df.groupby("label", group_keys=False)
                     .sample(n=min(SMOKE_VAL_PER_CLASS, int(val_df['label'].value_counts().min())),
                             random_state=SEED)
                     .sample(frac=1, random_state=SEED).reset_index(drop=True))
    ood_df = None; fp_df = None
    log(f"⚡ smoke val: {len(val_df)} mẫu cân bằng; bỏ qua OOD và test_fp")

# ── Tokenizer + HF Dataset ───────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_hf(df):
    ds = Dataset.from_pandas(df[["text", "label", "sample_weight"]], preserve_index=False)
    def tok(b):
        enc = tokenizer(b["text"], truncation=True, max_length=MAX_LEN)
        enc["labels"] = b["label"]
        enc["sample_weight"] = [float(w) for w in b["sample_weight"]]
        return enc
    return ds.map(tok, batched=True, remove_columns=ds.column_names)

train_ds = to_hf(train_df)
val_ds = to_hf(val_df) if val_df is not None else None
if SMOKE_TEST:
    train_ds = train_ds.shuffle(seed=SEED).select(range(min(SMOKE_N, len(train_ds))))
log("train_ds:", train_ds, "| val:", val_ds is not None)

# ── WeightedTrainer + collator + model ───────────────────────────
class WeightedTrainer(Trainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # compute_loss lấy mean theo micro-batch, không dùng num_items_in_batch.
        # Ép False để Trainer tự chia đúng theo grad accumulation (DDP tự average giữa process).
        self.model_accepts_loss_kwargs = False

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        weights = inputs.pop("sample_weight", None)
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss = nn.CrossEntropyLoss(reduction="none")(logits.view(-1, logits.size(-1)), labels.view(-1))
        if weights is not None:
            loss = loss * weights.to(loss.device)
        loss = loss.mean()
        return (loss, outputs) if return_outputs else loss

def collate(features):
    weights = torch.tensor([f.pop("sample_weight") for f in features], dtype=torch.float)
    labels  = torch.tensor([f.pop("labels") for f in features], dtype=torch.long)
    batch = tokenizer.pad(features, padding=True, pad_to_multiple_of=8, return_tensors="pt")
    batch["labels"] = labels
    batch["sample_weight"] = weights
    return batch

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2, **MODEL_KWARGS)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    scores = logits[:, 1] - logits[:, 0]
    n_bad = int(np.sum(~np.isfinite(scores)))
    if n_bad:
        log(f"⚠️  {n_bad}/{len(scores)} logits NaN/Inf -> thay = 0")
        scores = np.nan_to_num(scores, nan=0.0, posinf=1e4, neginf=-1e4)
    return {"auroc": float(roc_auc_score(labels, scores))} if len(np.unique(labels)) == 2 else {"auroc": 0.0}

callbacks = []
if val_ds is None:
    eval_kw = dict(eval_strategy="no", save_strategy="no")
elif SMOKE_TEST:
    eval_kw = dict(eval_strategy="steps", eval_steps=max(10, SMOKE_MAX_STEPS // 2), save_strategy="no")
else:
    eval_kw = dict(eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
                   load_best_model_at_end=True, metric_for_best_model="auroc", greater_is_better=True)
    callbacks = [EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PATIENCE)]

args = TrainingArguments(
    output_dir=str(OUT_DIR / "ckpt"),
    num_train_epochs=EPOCHS,
    max_steps=(SMOKE_MAX_STEPS if SMOKE_TEST else -1),
    per_device_train_batch_size=PER_DEVICE_BATCH,
    per_device_eval_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=grad_accum,
    optim=OPTIM,
    learning_rate=LEARNING_RATE,
    adam_epsilon=ADAM_EPSILON,
    weight_decay=0.01,
    warmup_ratio=WARMUP_RATIO,
    max_grad_norm=MAX_GRAD_NORM,
    fp16=USE_FP16,
    gradient_checkpointing=GRAD_CKPT,
    gradient_checkpointing_kwargs=({"use_reentrant": False} if GRAD_CKPT else None),
    ddp_find_unused_parameters=False,   # seq-classification dùng hết param -> False nhanh hơn
    logging_steps=(10 if SMOKE_TEST else 50),
    report_to="none",
    dataloader_num_workers=2,
    dataloader_persistent_workers=True,
    group_by_length=False,             # ĐỒNG BỘ
    remove_unused_columns=False,
    seed=SEED,
    **eval_kw,
)
log(f"fp16={args.fp16}  grad_ckpt={args.gradient_checkpointing}  "
    f"world_size={args.world_size}  n_gpu(per proc)={args.n_gpu}  max_steps={args.max_steps}")

trainer = WeightedTrainer(model=model, args=args, train_dataset=train_ds,
                          eval_dataset=val_ds,
                          compute_metrics=compute_metrics if val_ds is not None else None,
                          data_collator=collate, callbacks=callbacks)
assert trainer.model_accepts_loss_kwargs is False
log("loss scaling fix: model_accepts_loss_kwargs=False ✅")

# ── Train (mọi rank tham gia) ────────────────────────────────────
import torch.distributed as dist
try:
    trainer.train()
except BaseException:
    if torch.cuda.is_available():
        print(f"[rank{RANK}] peak VRAM trước lỗi: "
              f"{torch.cuda.max_memory_allocated(LOCAL_RANK) / 1024**3:.2f} GiB",
              flush=True)
    # Tránh cảnh báo/leak ProcessGroupNCCL khi smoke OOM hoặc NaN làm một rank thoát.
    if dist.is_available() and dist.is_initialized():
        dist.destroy_process_group()
    raise

if torch.cuda.is_available():
    print(f"[rank{RANK}] peak VRAM: "
          f"{torch.cuda.max_memory_allocated(LOCAL_RANK) / 1024**3:.2f} GiB",
          flush=True)

# ── Từ đây chỉ RANK 0 làm (eval + save); rank khác dọn NCCL rồi thoát ─
if dist.is_available() and dist.is_initialized():
    dist.barrier()  # chờ mọi rank xong train (checkpoint save) trước khi rank phụ rời đi
if not IS_MAIN:
    if dist.is_available() and dist.is_initialized():
        dist.destroy_process_group()
    raise SystemExit(0)

DEVICE = f"cuda:{LOCAL_RANK}" if torch.cuda.is_available() else "cpu"
model = trainer.model.to(DEVICE).eval()
if not SMOKE_TEST:
    tokenizer.save_pretrained(OUT_DIR / "model"); model.save_pretrained(OUT_DIR / "model")
    print("✅ đã lưu model ->", OUT_DIR / "model", flush=True)
else:
    print("⚡ SMOKE: KHÔNG lưu model.", flush=True)

@torch.no_grad()
def predict_proba(texts, batch_size=PER_DEVICE_BATCH):
    model.eval(); scores = []; texts = list(texts)
    for i in range(0, len(texts), batch_size):
        enc = tokenizer(texts[i:i + batch_size], truncation=True, max_length=MAX_LEN,
                        padding=True, pad_to_multiple_of=8, return_tensors="pt").to(DEVICE)
        scores.extend(torch.softmax(model(**enc).logits, -1)[:, 1].float().cpu().tolist())
    return np.array(scores)

for d in [val_df, ood_df, fp_df]:
    if d is not None:
        d["score"] = predict_proba(d["text"].tolist())

_std = float(np.std(val_df["score"])) if val_df is not None else 0.0
_auc = roc_auc_score(val_df["label"], val_df["score"]) if val_df is not None else float("nan")
print(f"std(score val) = {_std:.4f} | AUROC(val) = {_auc:.4f}", flush=True)
if _std < 1e-3 or _auc < 0.55:
    print("🛑 CẢNH BÁO: model gần như KHÔNG học (output hằng số / AUROC~0.5).", flush=True)

def tpr_at_fpr(y, s, fpr):
    y, s = np.asarray(y), np.asarray(s); neg = s[y == 0]
    if len(neg) == 0 or (y == 1).sum() == 0: return np.nan, np.nan
    thr = np.quantile(neg, 1 - fpr, method="higher")
    return float((s[y == 1] >= thr).mean()), float(thr)

def auroc(y, s):
    y = np.asarray(y)
    return float(roc_auc_score(y, s)) if len(np.unique(y)) == 2 else np.nan

_, TAU = tpr_at_fpr(val_df["label"], val_df["score"], TARGET_FPR)
print(f"τ (FPR={TARGET_FPR:.0%} trên val) = {TAU:.4f}", flush=True)

def report_split(name, d):
    if d is None: return None
    y, s = d["label"].values, d["score"].values
    row = {"split": name, "n": len(d), "n_ai": int((y == 1).sum()),
           "auroc": auroc(y, s),
           "auprc": float(average_precision_score(y, s)) if len(np.unique(y)) == 2 else np.nan}
    for f in [0.001, 0.01, 0.05]:
        row[f"tpr@fpr{f}"] = tpr_at_fpr(y, s, f)[0]
    row["tpr@tau"] = float((s[y == 1] >= TAU).mean()) if (y == 1).any() else np.nan
    row["fpr@tau"] = float((s[y == 0] >= TAU).mean()) if (y == 0).any() else np.nan
    return row

rows = [r for r in [report_split("val", val_df), report_split("test_ood_model", ood_df),
                    report_split("test_fp", fp_df)] if r]

# ── ĐÁNH GIÁ ĐỐI-SHORTCUT trên TẬP MATCHED (make_matched_eval.py) ─────
# Đây là con số trả lời phản biện "0.99 của anh chỉ là đếm chữ với độ dài từ".
#
# Tập matched là tập con mà MỌI mô hình chỉ-đặc-trưng-bề-mặt đều về mức ngẫu nhiên —
# đã kiểm bằng hai mô hình để không lập luận vòng tròn (fit trên train, và fit LẠI
# out-of-fold trên chính tập con):
#     val            876 -> 296 dòng    surface 0.9049 -> 0.5027 / 0.5475
#     test_ood_model 3.028 -> 1.206     surface 0.8715 -> 0.5446 / 0.4663
# Nên nếu detector vẫn giữ AUROC cao ở đây thì phần đó KHÔNG đến từ đặc trưng bề mặt.
#
# Sàn để so: GBM 22 đặc trưng bề mặt đạt AUROC 0.8715 trên test_ood_model đầy đủ
# nhưng 0.5446 trên tập matched (surface_baseline.py). Khoảng cách giữa detector và
# sàn đó là phần detector thực sự đóng góp.
#
# Chấm lại trực tiếp trên text của tập matched, KHÔNG join theo chỉ số dòng: tập
# matched chỉ giữ ~1/3 dòng và thứ tự khác, join theo vị trí là sai âm thầm.
# Tra globals() thay vì gọi thẳng tên: nhánh supervised có predict_proba() và
# SMOKE_TEST, nhánh RADAR dùng detector_scores() và SMOKE. Gọi thẳng thì block này
# chạy được ở nhánh này và NameError ở nhánh kia — mà lỗi đó chỉ nổ ở CUỐI buổi
# train, sau khi đã tốn hết GPU-hour.
_score_fn = globals().get("predict_proba") or globals().get("detector_scores")
_smoke = bool(globals().get("SMOKE_TEST") or globals().get("SMOKE") or False)
assert _score_fn is not None, "không tìm thấy hàm chấm điểm (predict_proba/detector_scores)"

for _sp in ("val", "test_ood_model"):
    _mp = DATA_DIR / f"matched_eval_{_sp}_{DATASET_TAG}.parquet"
    if not _mp.exists():
        print(f"  (thiếu {_mp.name} -> bỏ qua đánh giá matched cho {_sp})", flush=True)
        continue
    _m = pd.read_parquet(_mp)
    _m["score"] = _score_fn(_m["text"].tolist())
    _row = report_split(f"{_sp}_matched", _m)
    if _row:
        rows.append(_row)
        print(f"  matched {_sp}: n={len(_m)} AUROC={_row['auroc']:.4f}", flush=True)
    if not _smoke:
        _keep = [c for c in ("sample_id", "paper_id", "label", "text_role",
                             "surface_propensity", "score") if c in _m.columns]
        _m[_keep].to_parquet(OUT_DIR / f"pred_matched_{_sp}.parquet", index=False)
metrics_df = pd.DataFrame(rows).set_index("split")
pd.set_option("display.width", 200, "display.max_columns", 20)
print(metrics_df.round(4).to_string(), flush=True)

val_auc = metrics_df.loc["val", "auroc"] if "val" in metrics_df.index else np.nan
ood_auc = metrics_df.loc["test_ood_model", "auroc"] if "test_ood_model" in metrics_df.index else np.nan
print("=" * 60); print(f"HEADLINE — {RUN_NAME} (DDP dual-T4)"); print("=" * 60)
if "test_ood_model" in metrics_df.index:
    print(f"1) TPR@FPR=1% OOD-model : {metrics_df.loc['test_ood_model','tpr@fpr0.01']:.3f}")
if "test_fp" in metrics_df.index:
    print(f"2) FPR thực @ τ test_fp : {metrics_df.loc['test_fp','fpr@tau']:.4f}  (mong ~{TARGET_FPR})")
print(f"3) AUROC gap val->OOD   : {val_auc:.3f} -> {ood_auc:.3f}  (Δ={val_auc-ood_auc:+.3f})", flush=True)

if not SMOKE_TEST:
    metrics_out = {"run": RUN_NAME, "model": MODEL_NAME, "tau": TAU, "target_fpr": TARGET_FPR,
                   "world_size": WORLD_SIZE, "seed": SEED, "metrics": metrics_df.reset_index().to_dict("records")}
    (OUT_DIR / "metrics.json").write_text(json.dumps(metrics_out, indent=2, default=float))
    for nm, d in [("val", val_df), ("test_ood_model", ood_df), ("test_fp", fp_df)]:
        if d is not None:
            d[["paper_id", "label", "ai_degree", "generator_family", "score"]].to_parquet(
                OUT_DIR / f"pred_{nm}.parquet", index=False)
    print("✅ đã lưu artifacts ->", OUT_DIR, flush=True)

# rank 0 dọn NCCL group -> hết warning "destroy_process_group() was not called"
if dist.is_available() and dist.is_initialized():
    dist.destroy_process_group()


## (Tuỳ chọn) Smoke test
Kiểm 2 GPU bắt tay + chặn NaN/OOM (~vài phút) **trước khi** chạy full. Bỏ comment dòng dưới để chạy.

In [ ]:
# # ⚡ SMOKE DDP (tùy chọn): kiểm 2 GPU bắt tay + ETA, ~vài phút. OK -> chạy cell FULL.
# !SMOKE=1 OMP_NUM_THREADS=2 torchrun --standalone --nproc_per_node=2 train_ddp.py

## FULL RUN — chạy nhiều seed (mean ± std)
Mỗi seed lưu riêng vào `artifacts_<model>_seed<N>/`. Sửa `SEEDS` nếu cần. Nếu Kaggle hết giờ, chạy bớt seed rồi chạy tiếp — cell tổng hợp tự gộp mọi seed đã có.

In [ ]:
# 🚀 FULL RUN — chạy NHIỀU SEED để báo cáo mean ± std (chuẩn paper)
# Mỗi seed lưu riêng vào artifacts_<run>_seed<SEED>/ ; sửa danh sách SEEDS nếu cần.
SEEDS = [42, 43, 44]
for s in SEEDS:
    print(f"\n================= SEED {s} =================")
    get_ipython().system(f'RUN_SEED={s} OMP_NUM_THREADS=2 torchrun --standalone --nproc_per_node=2 train_ddp.py')

## Tổng hợp kết quả qua các seed
Gộp mọi `artifacts_<model>_seed*/metrics.json` → **mean ± std** (dạng chép thẳng vào KLTN). Chạy sau khi có ≥1 seed.

In [ ]:
# 📊 TỔNG HỢP mean ± std qua các seed (chuẩn báo cáo paper)
import json, glob
import numpy as np, pandas as pd
from pathlib import Path

RUN_NAME = "deberta_v3_large"   # khớp CONFIG
paths = sorted(glob.glob(f"artifacts_{RUN_NAME}_seed*/metrics.json"))
print(f"Tìm thấy {len(paths)} seed cho {RUN_NAME}:", [Path(p).parent.name for p in paths])

rows = []
for p in paths:
    m = json.load(open(p))
    seed = Path(p).parent.name.split("seed")[-1]
    for r in m["metrics"]:
        rows.append({"seed": seed, "split": r["split"], "auroc": r.get("auroc"),
                      "tpr@fpr0.001": r.get("tpr@fpr0.001"), "tpr@fpr0.01": r.get("tpr@fpr0.01"),
                      "tpr@fpr0.05": r.get("tpr@fpr0.05"), "fpr@tau": r.get("fpr@tau")})
df = pd.DataFrame(rows)
if len(df) == 0:
    print("⚠️  Chưa có metrics — chạy ít nhất 1 seed ở cell FULL trước."); 
else:
    print(f"\n=== Chi tiết từng seed ({RUN_NAME}) ===")
    print(df.round(4).to_string(index=False))
    print("\n=== Report mean ± std qua {} seed (chép vào KLTN) ===".format(df['seed'].nunique()))
    def ms(s, col):
        v = pd.to_numeric(s[col], errors="coerce").dropna().values
        return f"{v.mean():.4f} ± {v.std(ddof=1) if len(v)>1 else 0.0:.4f}" if len(v) else "NaN"
    for split in ["val", "test_ood_model", "test_fp"]:
        s = df[df["split"] == split]
        if len(s) == 0: continue
        print(f"{split:16} AUROC={ms(s,'auroc')}  TPR@FPR1%={ms(s,'tpr@fpr0.01')}  "
              f"TPR@FPR0.1%={ms(s,'tpr@fpr0.001')}  FPR@tau={ms(s,'fpr@tau')}")
    out = Path(f"summary_{RUN_NAME}_multiseed.json")
    out.write_text(json.dumps(df.to_dict("records"), indent=2, default=float))
    print("\n✅ đã lưu", out)